In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Get the project root directory
PROJECT_ROOT = Path.cwd().parent

# Raw data directory
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"


print("Project Root:", PROJECT_ROOT)
print("Raw Data Path:", RAW_DATA_PATH)

Project Root: d:\cart2insights
Raw Data Path: d:\cart2insights\data\raw


In [3]:
from pathlib import Path
import pandas as pd

DATASETS = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv"
}

data = {}

print("=" * 80)
print("CHECKING RAW DATA FILES")
print("=" * 80)

for table_name, filename in DATASETS.items():

    file_path = RAW_DATA_PATH / filename

    if not file_path.exists():
        print(f"❌ MISSING : {filename}")
        continue

    file_size = file_path.stat().st_size

    if file_size == 0:
        print(f"❌ EMPTY   : {filename}  (0 bytes)")
        continue

    try:
        df = pd.read_csv(file_path, low_memory=False)

        if df.empty:
            print(f"⚠️ NO ROWS : {filename}")
        else:
            data[table_name] = df
            print(
                f"✅ OK      : {filename} "
                f"| Rows: {len(df):,} "
                f"| Columns: {len(df.columns)}"
            )

    except pd.errors.EmptyDataError:
        print(f"❌ EMPTY CSV: {filename}")

    except Exception as e:
        print(f"❌ ERROR   : {filename}")
        print(f"   {type(e).__name__}: {e}")

print("\n" + "=" * 80)
print(f"Successfully loaded: {len(data)} / {len(DATASETS)} datasets")
print("=" * 80)

CHECKING RAW DATA FILES
✅ OK      : olist_customers_dataset.csv | Rows: 99,441 | Columns: 5
✅ OK      : olist_geolocation_dataset.csv | Rows: 1,000,163 | Columns: 5
✅ OK      : olist_order_items_dataset.csv | Rows: 112,650 | Columns: 7
✅ OK      : olist_order_payments_dataset.csv | Rows: 103,886 | Columns: 5
✅ OK      : olist_order_reviews_dataset.csv | Rows: 99,224 | Columns: 7
✅ OK      : olist_orders_dataset.csv | Rows: 99,441 | Columns: 8
✅ OK      : olist_products_dataset.csv | Rows: 32,951 | Columns: 9
✅ OK      : olist_sellers_dataset.csv | Rows: 3,095 | Columns: 4
✅ OK      : product_category_name_translation.csv | Rows: 71 | Columns: 2

Successfully loaded: 9 / 9 datasets


In [4]:
summary = []

for table_name, df in data.items():
    summary.append({
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum()
    })

quality_summary = pd.DataFrame(summary)

display(quality_summary)

,table,rows,columns,duplicate_rows,missing_cells
0,customers,99441,5,0,0
1,geolocation,1000163,5,261831,0
2,order_items,112650,7,0,0
3,order_payments,103886,5,0,0
4,order_reviews,99224,7,0,145903
5,orders,99441,8,0,4908
6,products,32951,9,0,2448
7,sellers,3095,4,0,0
8,translation,71,2,0,0


In [5]:
missing_results = []

for table_name, df in data.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        missing_percentage = (missing_count / len(df)) * 100
        
        missing_results.append({
            "table": table_name,
            "column": column,
            "missing_count": missing_count,
            "missing_percentage": round(missing_percentage, 2)
        })

missing_df = pd.DataFrame(missing_results)

missing_df = missing_df[
    missing_df["missing_count"] > 0
].sort_values(
    ["table", "missing_percentage"],
    ascending=[True, False]
)

display(missing_df)

,table,column,missing_count,missing_percentage
25,order_reviews,review_comment_title,87656,88.34
26,order_reviews,review_comment_message,58247,58.70
35,orders,order_delivered_customer_date,2965,2.98
34,orders,order_delivered_carrier_date,1783,1.79
33,orders,order_approved_at,160,0.16
38,products,product_category_name,610,1.85
39,products,product_name_lenght,610,1.85
40,products,product_description_lenght,610,1.85
41,products,product_photos_qty,610,1.85
42,products,product_weight_g,2,0.01


In [6]:
duplicate_results = []

for table_name, df in data.items():

    duplicate_count = df.duplicated().sum()

    duplicate_percentage = (
        duplicate_count / len(df) * 100
    )

    duplicate_results.append({
        "table": table_name,
        "total_rows": len(df),
        "duplicate_rows": duplicate_count,
        "duplicate_percentage": round(
            duplicate_percentage, 2
        )
    })

duplicate_df = pd.DataFrame(duplicate_results)

display(duplicate_df)

,table,total_rows,duplicate_rows,duplicate_percentage
0,customers,99441,0,0.00
1,geolocation,1000163,261831,26.18
2,order_items,112650,0,0.00
3,order_payments,103886,0,0.00
4,order_reviews,99224,0,0.00
5,orders,99441,0,0.00
6,products,32951,0,0.00
7,sellers,3095,0,0.00
8,translation,71,0,0.00


In [7]:
PRIMARY_KEYS = {
    "customers": ["customer_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "translation": ["product_category_name"]
}

primary_key_results = []

for table_name, keys in PRIMARY_KEYS.items():

    df = data[table_name]

    for key in keys:

        if key not in df.columns:

            primary_key_results.append({
                "table": table_name,
                "column": key,
                "exists": False,
                "null_count": None,
                "unique": None
            })

            continue

        primary_key_results.append({
            "table": table_name,
            "column": key,
            "exists": True,
            "null_count": df[key].isna().sum(),
            "unique": df[key].is_unique
        })

primary_key_df = pd.DataFrame(primary_key_results)

display(primary_key_df)

,table,column,exists,null_count,unique
0,customers,customer_id,True,0,True
1,geolocation,geolocation_zip_code_prefix,True,0,False
2,order_items,order_id,True,0,False
3,order_items,order_item_id,True,0,False
4,order_payments,order_id,True,0,False
5,order_payments,payment_sequential,True,0,False
6,order_reviews,review_id,True,0,False
7,orders,order_id,True,0,True
8,products,product_id,True,0,True
9,sellers,seller_id,True,0,True


In [8]:
RELATIONSHIPS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id"),
    ("products", "product_category_name", "translation", "product_category_name")
]

foreign_key_results = []

for child_table, child_column, parent_table, parent_column in RELATIONSHIPS:

    child_df = data[child_table]
    parent_df = data[parent_table]

    parent_values = set(
        parent_df[parent_column].dropna().unique()
    )

    child_values = child_df[child_column].dropna()

    orphan_count = (
        ~child_values.isin(parent_values)
    ).sum()

    total_values = len(child_values)

    orphan_percentage = (
        orphan_count / total_values * 100
        if total_values > 0
        else 0
    )

    foreign_key_results.append({
        "child_table": child_table,
        "child_column": child_column,
        "parent_table": parent_table,
        "parent_column": parent_column,
        "orphan_records": orphan_count,
        "orphan_percentage": round(
            orphan_percentage, 2
        )
    })

foreign_key_df = pd.DataFrame(
    foreign_key_results
)

display(foreign_key_df)

,child_table,child_column,parent_table,parent_column,orphan_records,orphan_percentage
0,orders,customer_id,customers,customer_id,0,0.00
1,order_items,order_id,orders,order_id,0,0.00
2,order_items,product_id,products,product_id,0,0.00
3,order_items,seller_id,sellers,seller_id,0,0.00
4,order_payments,order_id,orders,order_id,0,0.00
5,order_reviews,order_id,orders,order_id,0,0.00
6,products,product_category_name,translation,product_category_name,13,0.04


In [9]:
range_results = []

numeric_checks = {
    "order_items": [
        "price",
        "freight_value"
    ],
    "order_payments": [
        "payment_value"
    ],
    "products": [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
}

for table_name, columns in numeric_checks.items():

    df = data[table_name]

    for column in columns:

        if column not in df.columns:
            continue

        negative_count = (
            df[column] < 0
        ).sum()

        range_results.append({
            "table": table_name,
            "column": column,
            "negative_values": negative_count
        })

range_df = pd.DataFrame(range_results)

display(range_df)

,table,column,negative_values
0,order_items,price,0
1,order_items,freight_value,0
2,order_payments,payment_value,0
3,products,product_weight_g,0
4,products,product_length_cm,0
5,products,product_height_cm,0
6,products,product_width_cm,0


In [10]:
reviews = data["order_reviews"]

invalid_review_scores = reviews[
    ~reviews["review_score"].isin([1, 2, 3, 4, 5])
    & reviews["review_score"].notna()
]

print(
    "Invalid review scores:",
    len(invalid_review_scores)
)

display(invalid_review_scores.head())

Invalid review scores: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [11]:
orders = data["orders"].copy()

DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

date_quality_results = []

for column in DATE_COLUMNS:

    converted = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

    invalid_count = (
        converted.isna() &
        orders[column].notna()
    ).sum()

    date_quality_results.append({
        "column": column,
        "original_dtype": str(
            orders[column].dtype
        ),
        "invalid_dates": invalid_count
    })

date_quality_df = pd.DataFrame(
    date_quality_results
)

display(date_quality_df)

,column,original_dtype,invalid_dates
0,order_purchase_timestamp,str,0
1,order_approved_at,str,0
2,order_delivered_carrier_date,str,0
3,order_delivered_customer_date,str,0
4,order_estimated_delivery_date,str,0


In [12]:
def detect_iqr_outliers(df, column):

    series = df[column].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (series < lower_bound) |
        (series > upper_bound)
    ).sum()

    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count
    }

In [13]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Quality reports folder
QUALITY_REPORT_PATH = (
    PROJECT_ROOT
    / "data"
    / "quality_reports"
)

# Create the folder if it does not exist
QUALITY_REPORT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Quality report folder:")
print(QUALITY_REPORT_PATH)

Quality report folder:
d:\cart2insights\data\quality_reports


In [14]:
# ============================================================
# CELL 18 - CREATE AND SAVE ALL QUALITY REPORTS
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Define project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

QUALITY_REPORT_PATH = (
    PROJECT_ROOT
    / "data"
    / "quality_reports"
)

QUALITY_REPORT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Create dataset summary
# ------------------------------------------------------------

dataset_summary = []

for table_name, df in data.items():

    dataset_summary.append({
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum()
    })

dataset_summary_df = pd.DataFrame(dataset_summary)

# ------------------------------------------------------------
# 3. Create missing-value report
# ------------------------------------------------------------

missing_results = []

for table_name, df in data.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        missing_percentage = (
            missing_count / len(df) * 100
        )

        missing_results.append({
            "table": table_name,
            "column": column,
            "missing_count": missing_count,
            "missing_percentage": round(
                missing_percentage, 2
            )
        })

missing_df = pd.DataFrame(missing_results)

# ------------------------------------------------------------
# 4. Create duplicate report
# ------------------------------------------------------------

duplicate_results = []

for table_name, df in data.items():

    duplicate_count = df.duplicated().sum()

    duplicate_results.append({
        "table": table_name,
        "total_rows": len(df),
        "duplicate_rows": duplicate_count,
        "duplicate_percentage": round(
            duplicate_count / len(df) * 100,
            2
        )
    })

duplicate_df = pd.DataFrame(duplicate_results)

# ------------------------------------------------------------
# 5. Save reports that already exist
# ------------------------------------------------------------

reports = {
    "dataset_summary.csv": dataset_summary_df,
    "missing_values_report.csv": missing_df,
    "duplicate_report.csv": duplicate_df,
}

# Save additional reports only if they exist
optional_reports = {
    "primary_key_report.csv": "primary_key_df",
    "foreign_key_report.csv": "foreign_key_df",
    "range_check_report.csv": "range_df",
    "date_quality_report.csv": "date_quality_df",
    "outlier_report.csv": "outlier_df",
}

for filename, variable_name in optional_reports.items():

    if variable_name in globals():
        reports[filename] = globals()[variable_name]

# ------------------------------------------------------------
# 6. Write all reports
# ------------------------------------------------------------

for filename, dataframe in reports.items():

    dataframe.to_csv(
        QUALITY_REPORT_PATH / filename,
        index=False
    )

    print(f"✅ Saved: {filename}")

print("\n" + "=" * 70)
print("QUALITY REPORTS CREATED SUCCESSFULLY")
print("=" * 70)

print("Location:")
print(QUALITY_REPORT_PATH)

print("\nReports:")
for filename in reports:
    print(f" - {filename}")

✅ Saved: dataset_summary.csv
✅ Saved: missing_values_report.csv
✅ Saved: duplicate_report.csv
✅ Saved: primary_key_report.csv
✅ Saved: foreign_key_report.csv
✅ Saved: range_check_report.csv
✅ Saved: date_quality_report.csv

QUALITY REPORTS CREATED SUCCESSFULLY
Location:
d:\cart2insights\data\quality_reports

Reports:
 - dataset_summary.csv
 - missing_values_report.csv
 - duplicate_report.csv
 - primary_key_report.csv
 - foreign_key_report.csv
 - range_check_report.csv
 - date_quality_report.csv
